In [5]:
import pandas as pd
import numpy as np

raw_path = '../01_raw_data/'
clean_path = '../02_cleaned_data/'

# Load the 5 datasets
orders = pd.read_csv(raw_path + 'olist_orders_dataset.csv')
items = pd.read_csv(raw_path + 'olist_order_items_dataset.csv')
customers = pd.read_csv(raw_path + 'olist_customers_dataset.csv')
products = pd.read_csv(raw_path + 'olist_products_dataset.csv')
translations = pd.read_csv(raw_path + 'product_category_name_translation.csv')

print("Row & Column Counts:")
print(f"Orders:    {orders.shape}")
print(f"Items:     {items.shape}")
print(f"Customers: {customers.shape}")
print(f"Products:  {products.shape}")

print("\nMissing Values in Orders:")
print(orders.isnull().sum())

Row & Column Counts:
Orders:    (99441, 8)
Items:     (112650, 7)
Customers: (99441, 5)
Products:  (32951, 9)

Missing Values in Orders:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [7]:
# Convert date strings to proper datetime objects
date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# 1. Calculate raw delivery speed and delay in days first
orders['actual_delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

# 2. Use NumPy np.where BEFORE converting to nullable Int64
orders['delivery_sla_status'] = np.where(
    orders['order_delivered_customer_date'].isna(), 'Not Delivered',
    np.where(orders['delivery_delay_days'] > 0, 'Delayed', 'On Time')
)

# 3. Now convert to nullable Int64 so SQL Server imports clean integers without .0 decimals
orders['actual_delivery_days'] = orders['actual_delivery_days'].astype('Int64')
orders['delivery_delay_days'] = orders['delivery_delay_days'].astype('Int64')

orders = orders.drop_duplicates(subset=['order_id'])
print(orders['delivery_sla_status'].value_counts())

delivery_sla_status
On Time          89941
Delayed           6535
Not Delivered     2965
Name: count, dtype: int64


In [8]:
# 1. Translate product categories to English & handle missing values
products = products.merge(translations, on='product_category_name', how='left')
products['category_name'] = products['product_category_name_english'].fillna('uncategorized')

products_clean = products[['product_id', 'category_name', 'product_weight_g']].copy()
products_clean['category_name'] = products_clean['category_name'].str.replace('_', ' ').str.title()
products_clean['product_weight_g'] = products_clean['product_weight_g'].fillna(0).astype('Int64')

# 2. Clean Order Items & compute total item value using NumPy
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')
items['total_item_value'] = np.round(items['price'] + items['freight_value'], 2)

# 3. Clean Customer location strings
customers['customer_city'] = customers['customer_city'].str.strip().str.title()
customers['customer_state'] = customers['customer_state'].str.strip().str.upper()

print("Top 5 Product Categories:")
print(products_clean['category_name'].value_counts().head(5))

Top 5 Product Categories:
category_name
Bed Bath Table     3029
Sports Leisure     2867
Furniture Decor    2657
Health Beauty      2444
Housewares         2335
Name: count, dtype: int64


In [9]:
orders.to_csv(clean_path + 'clean_orders.csv', index=False, date_format='%Y-%m-%d %H:%M:%S')
items.to_csv(clean_path + 'clean_order_items.csv', index=False, date_format='%Y-%m-%d %H:%M:%S')
customers.to_csv(clean_path + 'clean_customers.csv', index=False)
products_clean.to_csv(clean_path + 'clean_products.csv', index=False)

print("SUCCESS: 4 cleaned CSVs saved in 02_cleaned_data!")

SUCCESS: 4 cleaned CSVs saved in 02_cleaned_data!
